# PIQP-style QP solver: demo

Solves `min 1/2 x'Px + c'x  s.t.  Ax = b,  Gx <= h` with the solver in `piqp_solver.py`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from piqp_solver import piqp

## A 2-D example

Minimize the distance to the point (2, 2) inside the box `0 <= x <= 1.5` with the extra constraint `x1 + 2 x2 <= 2.5`.

In [ ]:
P = 2 * np.eye(2)
c = -2 * np.array([2., 2.])          # |x - (2,2)|^2 up to a constant
A, b = np.zeros((0, 2)), np.zeros(0)  # no equality constraints
G = np.array([[1., 0], [0, 1], [-1, 0], [0, -1], [1, 2]])
h = np.array([1.5, 1.5, 0, 0, 2.5])

res = piqp((P, c, A, b, G, h), verbose=True)
print(res.status, 'after', res.iterations, 'iterations')
print('x* =', res.x)
print('active constraints:', np.where(res.z > 1e-6)[0])

In [ ]:
xx, yy = np.meshgrid(np.linspace(-0.5, 2.5, 300), np.linspace(-0.5, 2.5, 300))
obj = (xx - 2)**2 + (yy - 2)**2
feas = np.all([G[i, 0]*xx + G[i, 1]*yy <= h[i] for i in range(len(h))], axis=0)

plt.figure(figsize=(5, 5))
plt.contour(xx, yy, obj, levels=20, cmap='viridis', alpha=0.6)
plt.contourf(xx, yy, feas, levels=[0.5, 1], colors=['tab:blue'], alpha=0.2)
plt.plot(*res.x, 'r*', ms=15, label='PIQP solution')
plt.plot(2, 2, 'kx', label='unconstrained minimum')
plt.gca().set_aspect('equal'); plt.legend(); plt.title('Feasible set and solution')
plt.show()

## A random larger problem, checked against SciPy

In [ ]:
from scipy.optimize import minimize

rng = np.random.default_rng(0)
n, p, m = 30, 5, 50
M = rng.standard_normal((n, n)); P = M @ M.T + 1e-2 * np.eye(n)
c = rng.standard_normal(n)
A = rng.standard_normal((p, n)); G = rng.standard_normal((m, n))
x0 = rng.standard_normal(n); b = A @ x0; h = G @ x0 + rng.uniform(0.1, 1, m)

res = piqp((P, c, A, b, G, h))
ref = minimize(lambda x: 0.5*x@P@x + c@x, np.zeros(n), jac=lambda x: P@x + c, method='SLSQP',
               constraints=[{'type': 'eq', 'fun': lambda x: A@x - b},
                            {'type': 'ineq', 'fun': lambda x: h - G@x}],
               options={'ftol': 1e-12, 'maxiter': 500})
print(f"PIQP : {res.status}, {res.iterations} iterations, f = {0.5*res.x@P@res.x + c@res.x:.10f}")
print(f"SciPy: f = {ref.fun:.10f}")